# Sketch2Image: Stable Diffusion 1.5 + ControlNet

I ran this notebook on Google Colab with a T4 GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

In [ ]:
import os, subprocess
REPO = "https://github.com/MarufDzine/sketch2image.git"
# clone only once (re-running this cell does not clone again)
if not os.path.exists("sketch2img.py"):
    if not os.path.exists("sketch2image"):
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    os.chdir("sketch2image")
print(os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

## 2. My sketches

In [ ]:
!ls sketches

## 3. Preprocessing check
What the model sees for each sketch and each control type.

In [ ]:
import yaml, matplotlib.pyplot as plt
from sketch2img import prepare
cfg = yaml.safe_load(open("config.yaml"))
controls = ["scribble", "lineart", "canny"]
fig, axes = plt.subplots(len(cfg["sketches"]), 4, figsize=(12, 3.2 * len(cfg["sketches"])), squeeze=False)
for i, sk in enumerate(cfg["sketches"]):
    for j, c in enumerate(controls):
        img, lines, ctrl = prepare(sk["file"], c)
        if j == 0:
            axes[i, 0].imshow(img); axes[i, 0].set_title(sk["file"])
        axes[i, j + 1].imshow(ctrl); axes[i, j + 1].set_title(c)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Quick test on one sketch

In [ ]:
!python sketch2img.py --sketch sketches/house.jpeg --prompt "a small house next to a pine tree at night, photorealistic, highly detailed" --out test.png
from IPython.display import Image, display
display(Image("test.png", width=400))

## 5. All experiments
Images that already exist are skipped, so I can re-run this after a disconnect.

In [ ]:
!python run_experiments.py

In [ ]:
!python make_figures.py

## 6. Results

In [ ]:
import glob
from IPython.display import Image, display, Markdown
for f in sorted(glob.glob("outputs/figures/*.png")):
    display(Markdown(f"**{f}**")); display(Image(f, width=1000))

In [ ]:
import pandas as pd
pd.read_csv("outputs/tables/summary.csv")

## 7. Save results
Only figures, tables, metrics and control images (the raw images are too many).

In [ ]:
!zip -qr results.zip outputs/figures outputs/tables outputs/results.csv outputs/control
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("results.zip is in the Output panel")

## 8. Web UI (Gradio)

In [ ]:
!python app.py --share